In [1]:
import os
import json
import duckdb
import hashlib

# Setup paths
FHIR_DIR = "../synthea/output/fhir"
DB_PATH = "../data/omop_clinical.duckdb"

def stable_person_id(source_id: str) -> int:
    """Generates a deterministic person_id using SHA-256 for perfect relational integrity."""
    return int(hashlib.sha256(source_id.encode()).hexdigest(), 16) % (10**9)

def extract_conditions(patient_file):
    """Reads a FHIR bundle and extracts clinical conditions."""
    file_path = os.path.join(FHIR_DIR, patient_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        fhir_data = json.load(f)
        
    conditions = []
    
    for entry in fhir_data.get('entry', []):
        resource = entry.get('resource', {})
        
        if resource.get('resourceType') == 'Condition':
            # Extract stable Patient ID
            subject_ref = resource.get('subject', {}).get('reference', '')
            patient_source_id = subject_ref.replace('urn:uuid:', '')
            person_id = stable_person_id(patient_source_id)
            
            # Extract Condition Data
            code_block = resource.get('code', {}).get('coding', [{}])[0]
            snomed_code = code_block.get('code', '0')
            condition_text = code_block.get('display', 'Unknown')
            
            start_date = resource.get('onsetDateTime', '1900-01-01')[:10] 
            
            conditions.append((person_id, snomed_code, condition_text, start_date))
            
    return conditions

# EXECUTION BLOCK
print("⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION) [V3: SEMANTIC RIGOR]\n" + "-"*50)

print("🔍 Extracting conditions from FHIR JSON files...")
json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
all_conditions = []

for file in json_files:
    all_conditions.extend(extract_conditions(file))

print(f"📊 Extracted {len(all_conditions)} raw condition records.")
print("🔌 Connecting to DuckDB for standardized insertion...")

try:
    with duckdb.connect(DB_PATH) as con:
        # Create staging table
        con.execute("DROP TABLE IF EXISTS stg_condition")
        con.execute("""
            CREATE TEMPORARY TABLE stg_condition (
                person_id BIGINT,
                snomed_code VARCHAR,
                condition_text VARCHAR,
                start_date DATE
            )
        """)
        
        con.executemany("INSERT INTO stg_condition VALUES (?, ?, ?, ?)", all_conditions)
        
        # Ensure target table exists
        con.execute("""
            CREATE TABLE IF NOT EXISTS condition_occurrence (
                condition_occurrence_id BIGINT PRIMARY KEY,
                person_id BIGINT,
                condition_concept_id INTEGER,
                condition_start_date DATE,
                condition_source_value VARCHAR,
                condition_source_concept_id INTEGER
            )
        """)
        
        # Idempotency: Clear existing records to prevent duplication
        con.execute("DELETE FROM condition_occurrence")
        
        # Insertion with strict OMOP semantic separation
        con.execute("""
            INSERT INTO condition_occurrence 
            SELECT 
                ROW_NUMBER() OVER () AS condition_occurrence_id,
                stg.person_id,
                COALESCE(c.concept_id, 0) AS condition_concept_id, -- Standard Concept for Analysis
                stg.start_date AS condition_start_date,
                stg.condition_text AS condition_source_value,
                0 AS condition_source_concept_id -- explicitly set to 0 as the source text is unmapped
            FROM stg_condition stg
            LEFT JOIN concept c 
                ON stg.snomed_code = c.concept_code 
                AND c.vocabulary_id = 'SNOMED'
                AND c.domain_id = 'Condition'
        """)
        
        mapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id != 0").fetchone()[0]
        unmapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id = 0").fetchone()[0]
        
        print("\n✅ ETL Complete!")
        print(f" - Successfully mapped (Exact Match): {mapped_count} conditions")
        print(f" - Sent to AI Fallback Queue (ID 0): {unmapped_count} conditions")

except Exception as e:
    print(f"❌ Database error: {e}")

⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION) [V3: SEMANTIC RIGOR]
--------------------------------------------------
🔍 Extracting conditions from FHIR JSON files...
📊 Extracted 1758 raw condition records.
🔌 Connecting to DuckDB for standardized insertion...

✅ ETL Complete!
 - Successfully mapped (Exact Match): 680 conditions
 - Sent to AI Fallback Queue (ID 0): 1078 conditions
